# Survey non-response: the patients who never answered

This notebook asks the navigation question of the
[point-treatment tutorial](point-treatment-tmle.ipynb) again, with one new fact. About a quarter of
patients never return the 30-day transition survey. Each step shows its code, its output, and what
the output tells you.
[Missing outcomes](../technical-reference/point-treatment-tmle.md#missing-outcomes-and-controlled-direct-effects)
gives the clever covariate and the identification argument.

## The applied question

The survey goes out 30 days after discharge and closes on day 45. The program sponsor still wants
the average treatment effect (ATE) for **every eligible patient**, not only for respondents.
Discharge risk and prior utilization affect response, and navigation can change willingness to
reply.

The response model conditions on the arm and the baseline adjustment set. A baseline predictor of
response, such as language, must enter that set. Contact attempts after assignment cannot enter it,
because navigation can change them. The protocol scores death before day 30 as the worst transition
score (composite strategy). That score counts as observed, so only living patients who do not
respond are missing.

## What you will learn

| after this notebook you can | the step that shows it |
| --- | --- |
| see how respondents differ from the eligible population | Step 3 |
| record the response rule in the protocol and the design | Steps 4 and 5 |
| fit TMLE with a response model, and read its interval | Step 6 |
| say what a complete-case fit estimates, and when it agrees | Steps 7 and 8 |
| report a top-box rate as a difference, a risk ratio, and an odds ratio | Step 9 |
| recognize a refused composition | Step 10 |
| read the support report and the missingness sensitivity curve | Steps 11 and 12 |

## Why this method

| your situation | what this method buys | what it costs |
| --- | --- | --- |
| outcomes missing for reasons recorded at baseline | the full-population estimand, identified under missingness at random given the baseline variables and the arm | a response model on top of the assignment model |
| response depends on the exposure | the two mechanisms compose into one factor in the clever covariate | identification needs the **product** of the two mechanisms to be positive |
| you want double robustness | you keep it, in a different shape | it becomes "the outcome regression is right, **or** the product of the assignment and response mechanisms is right" |

The table below defines the terms this notebook uses most. Each link goes to the canonical
definition.

| term | plain meaning |
| --- | --- |
| [estimand](../user-guide/estimands.md) | the number the question asks for, written before any model is chosen |
| [nuisance](../technical-reference/point-treatment-tmle.md) | a model the estimate needs but the question does not ask about. Here, the outcome regression Q, the treatment mechanism g, and the response mechanism |
| [missingness at random](../technical-reference/point-treatment-tmle.md#missing-outcomes-and-controlled-direct-effects) | among patients with the same arm and baseline profile, response says nothing more about the unobserved score |
| [targeting](../user-guide/methods-learners.md#targeting-and-bounds) | a small update to Q, weighted by g and the response mechanism, that removes first-order bias |
| [influence curve](../technical-reference/inference.md) | how much each row moves the estimate. Its variance gives the standard error |
| [positivity](../user-guide/results-assessment.md#diagnostics) | every kind of patient has some chance of each arm. Here, also some chance of responding |
| [double robustness](../technical-reference/point-treatment-tmle.md) | the point estimate stays consistent when either side of the "or" above is consistent |

## Step 1: set up

The setup imports the learners and prints the installed `cleverly` version. Every fit below passes
its learners, fold count, and random seed explicitly, so a rerun reproduces the stored outputs.

In [1]:
import pandas as pd
from sklearn.linear_model import LinearRegression, LogisticRegression

import cleverly

pd.set_option("display.width", 130)
pd.set_option("display.max_columns", 20)
print("cleverly", cleverly.__version__)

cleverly 0.1.2


**What this output tells you.** The stored outputs in this notebook came from the version named
above. A different version can print different numbers.

## Step 2: the data

The code draws the data from `make_missing_outcome` and renames its columns to the program's names.
A `strength` of 2 adds outcome curvature, an effect that varies with discharge risk, and a stronger
link from discharge risk to response.

In [2]:
from cleverly.datasets import make_missing_outcome

program_names = {
    "Y": "transition_score",
    "A": "transition_navigation",
    "W1": "discharge_risk",
    "W2": "age",
    "W3": "prior_utilization",
    "Delta": "responded",
}
frame, truth = make_missing_outcome(n=4_000, seed=71, strength=2.0)
frame = frame.rename(columns=program_names)
print("rows and columns:", frame.shape)
print(frame.head().round(3))
print()
print(f"response rate:            {frame['responded'].mean():.3f}")
print(f"missing transition score: {int(frame['transition_score'].isna().sum())} rows")
print(f"population ATE:           {truth['ate']:.3f}")

rows and columns: (4000, 6)
   transition_score  transition_navigation  discharge_risk    age  prior_utilization  responded
0             2.476                    1.0           0.049  0.518             -0.110        1.0
1             2.841                    1.0          -0.089  0.503              0.511        1.0
2             2.195                    0.0           0.345 -1.421             -0.570        1.0
3             3.693                    1.0          -0.307  0.740              0.045        1.0
4               NaN                    0.0          -0.394  0.378              0.326        0.0

response rate:            0.756
missing transition score: 977 rows
population ATE:           1.200


**What this output tells you.** Each row is one discharge. `responded` is 1 when the survey came
back and 0 when it did not. Row 4 did not respond, so its `transition_score` is `NaN`. The
container refuses a missing outcome that carries no response indicator.

The response rate is 0.756, so 977 of the 4000 scores are missing. The true ATE for every eligible
patient is 1.200. Covariates are standardized (mean 0, SD 1), and scores are in synthetic units.

| feature of the law | what it means for the survey |
| --- | --- |
| response falls as discharge risk rises, and rises with prior utilization | respondents are not a random slice of the eligible population |
| response also depends on the arm | navigation changes who answers |
| the effect shrinks as discharge risk rises | respondents, who have lower risk, have a larger average effect than the eligible population |
| the outcome surface has curvature a main-effects model cannot reach | a linear regression fitted to respondents extrapolates the wrong shape |

`missing_outcome_dgp` in `cleverly.datasets` defines each of these features. A real program has no
`truth`. Every comparison against it below is a teaching device.

## Step 3: who answers the survey

Before any model, compare respondents with the patients who did not answer. The code prints the
mean of each baseline covariate by response status and the response rate in each arm. It then
computes the naive difference in mean score among respondents.

In [3]:
covariates = ["discharge_risk", "age", "prior_utilization"]
by_response = frame.groupby("responded")[covariates].mean()
print("mean baseline covariate by response status:")
print(by_response.round(3))
print()
response_by_arm = frame.groupby("transition_navigation")["responded"].mean()
print("response rate by arm:")
print(response_by_arm.round(3).to_string())
print()
respondents_only = frame[frame["responded"] == 1]
score_by_arm = respondents_only.groupby("transition_navigation")["transition_score"].mean()
unadjusted = score_by_arm.loc[1.0] - score_by_arm.loc[0.0]
print(f"unadjusted difference among respondents: {unadjusted:.3f}")
print(f"population ATE:                          {truth['ate']:.3f}")

mean baseline covariate by response status:
           discharge_risk    age  prior_utilization
responded                                          
0.0                 0.763  0.030             -0.150
1.0                -0.232 -0.018              0.038

response rate by arm:
transition_navigation
0.0    0.747
1.0    0.764

unadjusted difference among respondents: 1.768
population ATE:                          1.200


**What this output tells you.** Patients who did not respond have a mean `discharge_risk` of
0.763. Respondents have a mean of -0.232. The respondents are therefore a lower-risk group than the
eligible population.

The response rate is 0.747 under usual support and 0.764 under navigation. The marginal gap is
small, but the law's response model still contains the arm. The marginal rates mix that arm term
with the arms' different risk profiles.

The naive difference among respondents is 1.768, and the true ATE is 1.200. Two problems are mixed
in that gap. Discharge risk confounds the offer, and response selects a lower-risk group. The next
steps handle both.

## Step 4: write the protocol

A `StudyProtocol` records the scientific design before any model runs. This page starts from
`navigation_protocol()`, the protocol of the
[shared study design](index.md#the-shared-study-design). `dataclasses.replace` changes three fields
for the survey.

| field | what changes here |
| --- | --- |
| `target_population` | it says explicitly that non-respondents belong to the population |
| `outcome` | it names the survey that measures the score, sent on day 30 and closed on day 45 |
| `assumption_rationale` | it adds the response rule: missingness at random, and no post-assignment contact attempts in the response model |

In [4]:
from dataclasses import replace

from cleverly.datasets import navigation_protocol

program = navigation_protocol()
protocol = replace(
    program,
    target_population=f"{program.target_population}, whether or not they return the survey",
    outcome=f"{program.outcome} from the survey sent on day 30 and closed on day 45",
    assumption_rationale=(
        "The recorded baseline variables cover the measured common causes of the offer",
        "Given the arm and the baseline variables, survey response carries no further "
        "information about the unobserved score (missingness at random)",
        "Contact attempts after assignment stay out of the response model, "
        "because navigation can change them",
        *program.assumption_rationale[1:],
    ),
)
print("\n".join(protocol.summary_lines()))

causal study protocol: schema 1; de32ca40fa683035
target population: Adults with a discharge-home order at a participating hospital during the enrollment period, whether or not they return the survey
eligibility: ['Age 18 years or older', 'Discharge home ordered at a participating hospital']
time zero: Discharge-home order, after baseline measurement and before the navigation offer
treatment strategies: ['Offer standard transition navigation', 'Provide usual discharge support']
treatment versions: ['Bedside transition plan and two scheduled navigator contacts within 30 days', 'No access to the transition-navigation offer']
outcome: Patient-reported transition score from the survey sent on day 30 and closed on day 45
horizon: 30 days after discharge
intercurrent-event handling: ['Use the transition score regardless of readmission', 'Analyze the offer regardless of completed contacts', 'The protocol scores death before day 30 as the worst transition score (composite strategy)']
interfere

**What this output tells you.** The first line gives the fingerprint `de32ca40fa683035`. The fit in
Step 6 carries the same fingerprint. The other lines repeat each field.

Non-response is not an intercurrent event. It does not change what the score means. It hides a
score that exists. `StudyProtocol` has no field for a missing-data rule, so this page records the
rule in two places.

| where | what it records |
| --- | --- |
| `assumption_rationale` | why missingness at random is defensible, in words |
| `PointTreatment(missingness="responded")` in Step 5 | the column that marks an observed score, as a design role |

The synthetic law has three baseline covariates. The program adjustment set also names medication
burden and applicable site or calendar factors. A real protocol keeps them.

## Step 5: design and identification

The response indicator is a **design role**, like the outcome and the treatment. Declaring it tells
the estimator that the rows with a missing score are part of the population.

In [5]:
from cleverly import ATE, CausalStudy, PointTreatment

study = CausalStudy(
    frame,
    design=PointTreatment(
        outcome="transition_score",
        treatment="transition_navigation",
        adjustment=("discharge_risk", "age", "prior_utilization"),
        missingness="responded",
    ),
    protocol=protocol,
)
effect = study.identify(ATE(reference=0))

print(effect.summary())

average treatment effect, E[Y^a] - E[Y^reference]
identified by explicit-adjustment: E_W[E(transition_score | transition_navigation=a, responded=1, W)] - E_W[E(transition_score | transition_navigation=0, responded=1, W)] for a in [1]
adjustment/history: ['discharge_risk', 'age', 'prior_utilization']
required nuisances: ['outcome_regression', 'treatment_mechanism', 'missingness_mechanism']
assumptions:
  - consistency: Y = Y^a when A = a
  - no interference: one unit's potential outcome does not depend on other units' treatment assignments
  - no unmeasured confounding: Y^a is independent of A given W
  - positivity: P(transition_navigation = a | W) > 0 almost surely for every supported treatment level a in [0, 1]
  - missingness at random for responded: transition_score is independent of responded given (transition_navigation, W)
  - response positivity for responded: P(responded = 1 | transition_navigation = a, W) > 0 almost surely wherever the target functional evaluates arm a
causal

**What this output tells you.** The formula now conditions on `responded=1`. It fits the outcome
regression among respondents, then averages it over **all** patients. The required nuisances add
`missingness_mechanism` to the outcome regression and the treatment mechanism.

The summary adds two assumptions to the four of the point-treatment tutorial.

| assumption | what it means here | can the data check it? |
| --- | --- | --- |
| missingness at random | given the baseline adjustment set and the arm, response carries no further information about the unobserved score | no |
| response positivity | every kind of patient had some chance of both arms **and** of responding. Identification needs the product to be positive. Stable inference also needs it away from zero | partly, through the support report |

Missingness at random is not testable. A patient who ignores the survey because navigation failed
violates it directly. No diagnostic can detect that violation from the observed data. Step 12
stresses one declared departure instead.

## Step 6: estimate the ATE

The response model gets its own learner slot, `missingness_learner`.

| setting | value | what it does |
| --- | --- | --- |
| `outcome_learner` | linear regression | fits Q among respondents. It is wrong for this law |
| `treatment_learner` | main-effects logistic | fits g. It matches the law's assignment mechanism |
| `missingness_learner` | main-effects logistic | fits the response probability given the arm and the covariates. It matches the law's response mechanism |
| `CrossFitting(n_folds=5)` | five folds | predicts each row from models that did not see that row. [CV-TMLE](../technical-reference/cv-tmle.md) defines the construction |
| `Runtime(random_state=71, n_jobs=1)` | fixed seed, one process | makes the fit reproducible |

These learners deliberately use the second route to double robustness. The outcome model is wrong,
and the two mechanisms are right.

In [6]:
from cleverly import CrossFitting, ModelSpec, Runtime, TMLEMethod

method = TMLEMethod(
    models=ModelSpec(
        outcome_learner=LinearRegression(),
        treatment_learner=LogisticRegression(max_iter=1000),
        missingness_learner=LogisticRegression(max_iter=1000),
    ),
    cross_fitting=CrossFitting(n_folds=5),
    runtime=Runtime(random_state=71, n_jobs=1),
)
full = effect.estimate(method=method)

estimate = full["ate"]
print(full.summary())
print()
print(f"estimate:        {estimate.psi:.3f}")
print(f"standard error:  {estimate.std_error:.3f}")
print(f"95% CI:          ({estimate.ci[0]:.3f}, {estimate.ci[1]:.3f})")
print(f"population ATE:  {truth['ate']:.3f}")

Targeted maximum likelihood estimation
n = 4000 (3023 with an observed outcome); covariates = 3; P(A=1) = 0.4993
causal estimand: average treatment effect, E[Y^a] - E[Y^reference]
identification: explicit-adjustment; E_W[E(transition_score | transition_navigation=a, responded=1, W)] - E_W[E(transition_score | transition_navigation=0, responded=1, W)] for a in [1]
required nuisances: outcome_regression, treatment_mechanism, missingness_mechanism
identification assumptions: consistency: Y = Y^a when A = a; no interference: one unit's potential outcome does not depend on other units' treatment assignments; no unmeasured confounding: Y^a is independent of A given W; positivity: P(transition_navigation = a | W) > 0 almost surely for every supported treatment level a in [0, 1]; missingness at random for responded: transition_score is independent of responded given (transition_navigation, W); response positivity for responded: P(responded = 1 | transition_navigation = a, W) > 0 almost surely 

**What this output tells you.** The header reports 4000 rows, of which 3023 have an observed
outcome. The configuration lines show two truncation bounds. The propensity bound is
[0.009532, 0.9905], and the response probability bound is [0.01, 1].

The estimate is 1.185 with a standard error of 0.068. The 95% interval is (1.051, 1.319), and it
contains the true ATE of 1.200. That is one draw. One covered interval is not evidence of coverage.

## Step 7: the failure mode, complete cases estimate a different population

Now do what the program office would do without thinking about it. Drop the patients who never
answered, forget the indicator, and run the ordinary analysis with the same method. The code prints
both fits beside the true ATE.

The complete-case study carries no protocol. Its rows no longer match the protocol's target
population.

In [7]:
respondents = frame[frame["responded"] == 1].drop(columns=["responded"])
complete_case = (
    CausalStudy(
        respondents,
        design=PointTreatment(
            outcome="transition_score",
            treatment="transition_navigation",
            adjustment=("discharge_risk", "age", "prior_utilization"),
        ),
    )
    .identify(ATE(reference=0))
    .estimate(method=method)
)


def show(label, result, target):
    point = result["ate"]
    low, high = point.ci
    covered = low <= target <= high
    print(
        f"{label:22s} psi={point.psi:6.3f}  se={point.std_error:6.4f}  "
        f"CI=({low:.3f}, {high:.3f})  covers={covered}"
    )


show("complete cases only", complete_case, truth["ate"])
show("missingness declared", full, truth["ate"])
print(f"population ATE: {truth['ate']:.3f}")
print("rows used by the complete-case fit:", len(respondents), "of", len(frame))

complete cases only    psi= 1.467  se=0.0584  CI=(1.353, 1.581)  covers=False
missingness declared   psi= 1.185  se=0.0684  CI=(1.051, 1.319)  covers=True
population ATE: 1.200
rows used by the complete-case fit: 3023 of 4000


**What this output tells you.** The complete-case fit uses 3023 of 4000 rows. Its estimate is
1.467, above the population value of 1.200, and its interval (1.353, 1.581) excludes it. The fit
that declares the response mechanism covers the population value.

The complete-case fit averages over respondents only. Under missingness at random, it estimates the
effect standardized to the respondents' covariate distribution. That equals the eligible-population
ATE when the effect does not vary with the predictors of response. Here respondents have lower
discharge risk (Step 3) and a larger average effect.

Double robustness gives the complete-case fit no protection, because neither of its nuisance models
is correct among respondents.

| nuisance | why it is wrong among respondents |
| --- | --- |
| outcome regression | the linear model misses the law's curvature |
| treatment mechanism | response depends on the arm, so the probability of an offer among respondents is not a main-effects logistic function of the covariates |

The fit solved its score equation, but it answers a question about a different population.

## Step 8: a law where complete cases agree

This demonstration does not show that complete-case analysis is always wrong. At `strength=1.0` the
outcome is linear and the effect is constant. The code repeats the complete-case fit on that law.

In [8]:
mild_frame, mild_truth = make_missing_outcome(n=4_000, seed=71, strength=1.0)
mild_frame = mild_frame.rename(columns=program_names)
mild_respondents = mild_frame[mild_frame["responded"] == 1].drop(columns=["responded"])
mild = (
    CausalStudy(
        mild_respondents,
        design=PointTreatment(
            outcome="transition_score",
            treatment="transition_navigation",
            adjustment=("discharge_risk", "age", "prior_utilization"),
        ),
    )
    .identify(ATE(reference=0))
    .estimate(method=method)
)
show("mild law, complete cases", mild, mild_truth["ate"])
print(f"mild-law response rate:  {mild_frame['responded'].mean():.3f}")
print(f"mild-law population ATE: {mild_truth['ate']:.3f}")

mild law, complete cases psi= 1.224  se=0.0361  CI=(1.153, 1.294)  covers=True


mild-law response rate:  0.797
mild-law population ATE: 1.200


**What this output tells you.** The complete-case estimate is 1.224, and its interval
(1.153, 1.294) contains the mild-law ATE of 1.200. Both laws have the same population ATE. They
differ in the curvature and in how the effect varies with discharge risk.

The mild-law response rate is 0.797, against 0.756 in Step 2. That small difference does not
explain the change. On the mild law the respondents' average effect equals the population's, and
the linear model is correct. Under missingness at random, a correct outcome model identifies the
ATE from complete cases, as `missing_outcome_dgp` documents. Do not expect that agreement when
effects vary with response predictors.

## Step 9: reading it as a top-box rate

Program scorecards often report the share of patients above a declared transition threshold. They
also compare that share as a ratio. One shipped law carries both a binary outcome and a response
indicator, so the code checks those readings against a known truth.

The code prints each estimate and its interval. It also prints the distance from the estimate to
each interval limit, and the population value.

In [9]:
from cleverly import OddsRatio, RiskRatio
from cleverly.datasets import make_missing_outcome_binary

box_frame, box_truth = make_missing_outcome_binary(n=4_000, seed=72)
box_frame = box_frame.rename(columns={**program_names, "Y": "top_box"})
box_study = CausalStudy(
    box_frame,
    design=PointTreatment(
        outcome="top_box",
        treatment="transition_navigation",
        adjustment=("discharge_risk", "age", "prior_utilization"),
        missingness="responded",
    ),
)
box_method = TMLEMethod(
    models=ModelSpec(
        outcome_learner=LogisticRegression(max_iter=1000),
        treatment_learner=LogisticRegression(max_iter=1000),
        missingness_learner=LogisticRegression(max_iter=1000),
    ),
    cross_fitting=CrossFitting(n_folds=5),
    runtime=Runtime(random_state=72, n_jobs=1),
)
box_points = {}
for estimand, key in (
    (ATE(reference=0), "ate"),
    (RiskRatio(reference=0), "rr"),
    (OddsRatio(reference=0), "or"),
):
    point = box_study.identify(estimand).estimate(method=box_method)[key]
    box_points[key] = point
    low, high = point.ci
    print(
        f"{key}: {point.psi:6.4f}  CI=({low:.3f}, {high:.3f})  "
        f"below={point.psi - low:.3f}  above={high - point.psi:.3f}  "
        f"population {box_truth[key]:.4f}"
    )

ate: 0.1913  CI=(0.155, 0.228)  below=0.037  above=0.037  population 0.1862
rr: 1.5119  CI=(1.386, 1.650)  below=0.126  above=0.138  population 1.4947


or: 2.1772  CI=(1.866, 2.540)  below=0.311  above=0.363  population 2.1312


**What this output tells you.** The three rows are three readings of one comparison. Each interval
contains its population value on this draw.

| estimand | estimate | what it reports |
| --- | --- | --- |
| `ate` | 0.1913 | the difference in the top-box share, 19 percentage points |
| `rr` | 1.5119 | the multiplicative version a program scorecard uses |
| `or` | 2.1772 | the odds ratio, farther above one than the risk ratio |

For a common outcome, the gap between the odds ratio and the risk ratio is large. Reporting the odds
ratio as a risk ratio would overstate the change.

The ratio intervals are built on the log scale, so they are asymmetric. The `rr` interval reaches
0.126 below the estimate and 0.138 above it. The `ate` interval reaches 0.037 on each side.

## Step 10: a refused composition

`PopulationAttributableRisk` and `PopulationAttributableFraction` are refused under `missingness=`.
The code asks for the fraction and prints the refusal.

In [10]:
from cleverly import CapabilityError, PopulationAttributableFraction

try:
    box_study.identify(PopulationAttributableFraction(reference=0)).estimate(method=box_method)
except CapabilityError as error:
    refusal = str(error)
else:
    raise AssertionError("the attributable fraction should be refused under missingness=")
print("refused:", refusal)

refused: PopulationAttributableFraction does not yet support PointTreatment(missingness=...): under missingness at random the natural-course mean E[Y] needs an additional outcome/missingness score equation, and using complete cases would identify a different parameter. docs/roadmap.md F20 tracks this identification boundary.


**What this output tells you.** The refusal is a `CapabilityError` raised before any learner is
fitted. The message says that the natural-course mean needs an additional outcome and response score
equation. It also says that complete cases would identify a different parameter.

The 2026-09-12 source audit found no published derivation of that construction.
[RM8](../roadmap.md#rm8-missing-outcome-attributable-effects) records the audit, and
[F20](../roadmap.md#f20-missing-outcome-attributable-effects) states what a derivation must supply.

`NaturalCourseMean()` is supported under `missingness=`, with two contracts. Neither `method` nor
`box_method` on this page meets either contract. Both keep cross-fitting on, with folds stratified
by treatment.

| contract | outcome on this page it accepts | what the method needs |
| --- | --- | --- |
| ordinary TMLE | `transition_score` or `top_box` | `CrossFitting(enabled=False)`. The continuous `transition_score` also needs fixed `Targeting(q_bounds=...)` |
| stacked CV-TMLE | `top_box` only, because the contract requires a binary outcome | `CrossFitting(enabled=True, stratify_by="none")` with `n_folds` of at least 2 |

[Missing-outcome natural-course contracts](../technical-reference/scope-and-refusals.md#missing-outcome-natural-course-contracts)
lists every requirement and refusal for both contracts.

## Step 11: diagnostics, what the fit can check

The combined assessment of the Step 6 fit collects validation, diagnostics, and sensitivity in one
object. The option `include_retargets=True` adds the operations that retarget cached nuisance
predictions without a refit. The `arguments` mapping configures the two missingness operations,
which Step 12 reads.

The code prints the assessment summary and three retained reports: support, nuisance models, and
score equations.

In [11]:
assessment = full.assess(
    include_retargets=True,
    arguments={
        "missingness": {
            "gamma": (-2.0, -1.0, 0.0, 1.0, 2.0),
            "arm_gamma": {0: 0.0, 1: -1.0},
        },
        "tipping_gamma": {
            "arm_gamma": {0: 0.0, 1: -1.0},
        },
    },
)
print(assessment.summary())
print("needs attention:", tuple(item.name for item in assessment.attention))

support = assessment.report("support")
nuisance = assessment.report("nuisance_models")
scores = assessment.report("score_equations")
print()
print(support.summary())
print()
print(nuisance.summary())
print()
print(scores.summary())

Returned results
----------------
surface      operation            result                                                                                                                                                                                    
-----------  -------------------  ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
validation   support              maximum truncated fraction 0.0%; minimum effective-sample-size ratio 76.7%; group load: mean:h0 1001.0/3023 Kish-equivalent mask rows (33.1%; 25.0% all; draw 01 of 01); not estimator ESS
validation   nuisance_models      3 nuisance model report(s) are available                                                                                                                                                  
diagnostics  truncation_curve     1 parameter(s) over evaluated lower bounds [0.00

**What this output tells you.** Read the parts in order.

| output part | what it shows on this draw |
| --- | --- |
| `Checks` and `needs attention` | no row needs attention, and the score-equation check passed |
| positivity and overlap, first tables | the fitted treatment mechanism ranges from 0.1667 to 0.8364, and no unit is truncated |
| the `mechanism` table | the fitted response probability has a minimum of 0.0331. The product of the arm and response probabilities has a minimum of 0.0072 and an ESS / n of 0.767 |
| nuisance model diagnostics | calibration slopes near 1 for the propensity and response models, and an outcome `r2` of 0.5020 |
| score-equation check | targeting solved the composed score equation |

Positivity is now a statement about the product of two mechanisms. A patient with a middling chance
of navigation and of response can still have a small product. The clever covariate divides by that
product, and its largest absolute value here is 112.2.

The nuisance report prints "nuisance fits look reasonable", yet the outcome model is wrong by
construction. A held-out fit measure does not show that a nuisance model is correctly specified.

The summary also reports a robustness value of 0.146 for omitted confounding.
[Sensitivity analysis](../user-guide/results-assessment.md#sensitivity-analysis) explains that
quantity, and the [point-treatment tutorial](point-treatment-tmle.ipynb) reads it step by step.

## Step 12: sensitivity, what the fit cannot check

Missingness at random is not testable, so this step asks how far one stated departure must go to
change the conclusion. The missingness curve is a pattern-mixture sensitivity analysis. Within each
arm and covariate profile, it shifts the mean of the unobserved scores away from the respondents'
mean. [Missingness tilt and tipping gamma](../technical-reference/validation-methods.md#missingness-tilt-and-tipping-gamma)
defines both operations.

| element | meaning |
| --- | --- |
| `gamma` | the shift on the logit of the score rescaled to [0, 1]. The fit summary in Step 6 prints the rescaling range |
| `arm_gamma={0: 0.0, 1: -1.0}` | the control arm stays at missingness at random. Positive `gamma` lowers the unobserved navigation-arm mean |
| `gamma=0` | reproduces the missingness-at-random estimate by construction |
| tipping gamma | the smallest `gamma` at which the point estimate reaches zero |
| `ci_lower`, `ci_upper` | reuse the standard error of the fit, so they ignore uncertainty about `gamma` |

In [12]:
from scipy.special import expit

missingness_curve = assessment.report("missingness")
tipping_gamma = assessment.report("tipping_gamma")
print(missingness_curve.round(4).to_string(index=False))
print()
print(f"tipping gamma: {tipping_gamma:.3f}")
largest_shift = 0.5 - expit(-tipping_gamma)
print(f"largest shift of an unobserved navigation-arm mean at that tilt: {largest_shift:.3f}")

 gamma estimand     psi  std_err  ci_lower  ci_upper  is_mar  gamma[0]  gamma[1]
  -2.0      ate  2.6590   0.0684    2.5249    2.7930   False      -0.0       2.0
  -1.0      ate  2.0895   0.0684    1.9554    2.2235   False      -0.0       1.0
   0.0      ate  1.1852   0.0684    1.0512    1.3193    True       0.0      -0.0
   1.0      ate  0.2342   0.0684    0.1001    0.3682   False       0.0      -1.0
   2.0      ate -0.4175   0.0684   -0.5516   -0.2835   False       0.0      -2.0

tipping gamma: 1.299
largest shift of an unobserved navigation-arm mean at that tilt: 0.286


**What this output tells you.** The `gamma` = 0 row reproduces the Step 6 estimate, 1.1852. At
`gamma` = 1 the estimate falls to 0.2342, and at `gamma` = 2 it is -0.4175. Every row has the same
standard error, 0.0684, because the curve reuses the fitted one.

The tipping gamma is 1.299. At that tilt, an unobserved navigation-arm mean moves by at most 0.286 of
the score range. The largest move happens at mid-range. The ATE moves less, because respondents keep
their observed scores and a mean away from mid-range moves less.

This curve does not detect why patients did not answer. It shows how far one stated departure must
move before the conclusion changes. The review must decide whether a shift of that size is
plausible for this survey.

## How far to trust this

The [ordinary missing-outcome study](../technical-reference/method-evidence/ordinary-missing-outcome-tmle.md)
validates ordinary, non-cross-fitted TMLE with missing outcomes. It uses supplied oracle nuisances
on a binary law. No registered study covers the stacked cross-fitted construction or the linear
learners used on this page.

| layer | establishes | does not establish |
| --- | --- | --- |
| the support report | overlap for each fitted mechanism, plus joint ESS, concentration, and maximum leverage | that the response model is correct or missingness at random holds |
| the nuisance report | held-out fit and calibration measures for the treatment, response, and outcome models | that any nuisance model is correctly specified |
| the score-equation check | the targeting solved the composed score | that missingness at random holds |
| the missingness tilt and tipping gamma | estimate movement under one declared arm-specific departure | that the departure describes why patients did not respond |
| the mild-law comparison | complete cases agree when the effect is constant and the outcome model is correct | whether the same agreement holds in another population |
| the ordinary missing-outcome study | repeated-sampling truth, R `tmle` agreement, three-nuisance robustness, calibration, and a complete-case control for `ate`, `ey1`, and `ey0` | coverage for this continuous law, cross-fitting, or the `rr` and `or` readings, which keep only exact-law evidence under missing outcomes |

The strongest assumption on this page remains untestable. A survey whose non-response is driven by
the experience itself breaks missingness at random. The tilt shows consequences under one departure,
but it cannot establish which departure is plausible. Treat the estimate as conditional on an
argument about the mailing process.

## Where to go next

Non-response is one mechanism that removes patients from view. Plan exit is another, and it acts
over time rather than once. Read [time-to-event outcomes](longitudinal-survival.ipynb) for the version
where plan exit is the outcome.

The [examples index](index.md#the-program) lists every tutorial in the program.